# HTMLHeaderTextSplitter

[`MarkdownHeaderTextSplitter`](https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter) 와 개념적으로 유사한 `HTMLHeaderTextSplitter` 는 텍스트를 HTML 요소 수준에서 분할하고,
각 청크와 "관련된" 헤더 정보를 메타데이터로 추가하는 **구조 인식(structure-aware)** 분할기입니다.

요소별로 청크를 반환하거나 같은 메타데이터를 가진 요소를 합칠 수 있으며,
- (a) 관련 텍스트를 (대략적으로) 의미 단위로 묶고
- (b) 문서 구조에 담긴 맥락 정보를 보존하는 것을 목표로 합니다.

> **🔄 최신 버전 기준 변경 사항 (langchain-text-splitters 1.x)**
> - **`split_text_from_url()` 은 deprecated 되었습니다.** 리다이렉트를 통해 내부망·클라우드 메타데이터 주소에 접근할 수 있는
>   SSRF 취약점(GHSA-fv5p-p927-qmxr)이 발견되어 수정되었고, 이후 **HTML을 직접 받아와 `split_text()` 에 넘기는 방식**이 권장됩니다.
> - 내부 구현이 `lxml`/XSLT 기반에서 **BeautifulSoup** 기반으로 바뀌었으므로 `beautifulsoup4` 가 필요합니다.
> - 요소별 반환 옵션 `return_each_element` 예제를 추가했습니다.
> - 표·목록을 쪼개지 않는 **`HTMLSemanticPreservingSplitter`** 예제를 추가했습니다.
> - 공식 문서 링크를 현재 주소(docs.langchain.com)로 변경했습니다.

In [ ]:
%pip install -qU langchain-text-splitters beautifulsoup4 httpx

## HTML 문자열을 사용하는 경우

- `headers_to_split_on`: `(헤더 태그, 메타데이터 키)` 튜플 리스트
- `HTMLHeaderTextSplitter(headers_to_split_on=...)` 로 분할기를 만듭니다.

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>헤더1</h1>
        <p>헤더1 에 포함된 본문</p>
        <div>
            <h2>헤더2-1 제목</h2>
            <p>헤더2-1 에 포함된 본문</p>
            <h3>헤더3-1 제목</h3>
            <p>헤더3-1 에 포함된 본문</p>
            <h3>헤더3-2 제목</h3>
            <p>헤더3-2 에 포함된 본문</p>
        </div>
        <div>
            <h2>헤더2-2 제목2</h2>
            <p>헤더2-2 에 포함된 본문</p>
        </div>
        <br>
        <p>마지막 내용</p>
    </div>
</body>
</html>
"""

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
html_header_splits = html_splitter.split_text(html_string)

for header in html_header_splits:
    print(f"{header.page_content}")
    print(f"{header.metadata}", end="\n=====================\n")

### 🔄 추가: 요소별로 반환하기 (`return_each_element`)

기본값(`False`)은 같은 헤더 계층 아래의 요소들을 하나로 **합칩니다**. `True` 로 설정하면 각 HTML 요소가 별도의 `Document` 로 반환됩니다.

In [ ]:
element_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    return_each_element=True,
)
for doc in element_splitter.split_text(html_string):
    print(doc.page_content, "|", doc.metadata)

## 웹 URL에서 HTML을 로드하고 다른 splitter와 파이프라인으로 연결하기

🔄 **변경된 권장 방식**: `split_text_from_url(url)` 대신 HTTP 클라이언트로 HTML을 직접 받아와서 `split_text()` 에 넘깁니다.
이렇게 하면 타임아웃, 헤더(User-Agent), 인증, 재시도, 허용 도메인 검사 등을 애플리케이션이 직접 통제할 수 있습니다.

> ⚠️ 사용자가 입력한 URL을 그대로 요청하는 서비스라면, 요청 전에 허용 도메인(allowlist)을 검사하고 사설 IP·localhost 등으로의 요청을 막아야 합니다. (SSRF 방지)

In [ ]:
import httpx


def fetch_html(url: str, timeout: float = 30.0) -> str:
    """URL에서 HTML 문자열을 가져옵니다."""
    response = httpx.get(
        url,
        follow_redirects=True,
        timeout=timeout,
        headers={"User-Agent": "Mozilla/5.0 (compatible; langchain-tutorial)"},
    )
    response.raise_for_status()
    return response.text

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

url = "https://plato.stanford.edu/entries/goedel/"

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
    ("h4", "Header 4"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 🔄 split_text_from_url(url) → fetch_html(url) + split_text()
html_header_splits = html_splitter.split_text(fetch_html(url))

chunk_size = 500
chunk_overlap = 30
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

# 헤더 기준으로 나눈 Document들을 다시 크기 기준으로 분할 (헤더 메타데이터는 유지됨)
splits = text_splitter.split_documents(html_header_splits)

for header in splits[80:85]:
    print(f"{header.page_content}")
    print(f"{header.metadata}", end="\n=====================\n")

> 참고: 로컬 HTML 파일은 `html_splitter.split_text_from_file("path/to/file.html")` 로 분할할 수 있습니다.

## 한계

`HTMLHeaderTextSplitter` 는 HTML 문서 간 구조 차이를 처리하려고 하지만, 특정 헤더를 누락할 수 있습니다.

이 알고리즘은 헤더가 항상 관련 텍스트보다 "위"(이전 형제 노드, 조상 노드 및 그 조합)에 있다고 가정합니다.
다음 뉴스 기사(작성 시점 기준)에서는 최상위 헤드라인이 `h1` 으로 태그되어 있지만, 본문 텍스트와는 **별개의 하위 트리**에 있습니다.
따라서 `h1` 정보는 청크 메타데이터에 나타나지 않고, 해당되는 경우 `h2` 정보만 보입니다.

(오래된 기사라 페이지 구조가 바뀌었거나 접속이 차단될 수 있습니다.)

In [ ]:
url = "https://www.cnn.com/2023/09/25/weather/el-nino-winter-us-climate/index.html"

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
html_header_splits = html_splitter.split_text(fetch_html(url))

for header in html_header_splits:
    print(f"{header.page_content[:100]}")
    print(f"{header.metadata}", end="\n=====================\n")

## 🔄 추가: HTMLSemanticPreservingSplitter — 표·목록을 쪼개지 않는 분할

`HTMLHeaderTextSplitter` + `RecursiveCharacterTextSplitter` 조합은 **표나 목록을 중간에서 잘라** 표 헤더 같은 맥락을 잃을 수 있습니다.
`HTMLSemanticPreservingSplitter` 는 헤더 기준으로 나누면서 `elements_to_preserve` 에 지정한 요소(표, 목록 등)는 **통째로 유지**합니다.

- `max_chunk_size`: 목표 최대 크기. 단, 보존 요소를 다시 넣는 과정에서 이 크기를 **넘을 수 있습니다**(구조 보존 우선).
- `elements_to_preserve`: 분할하지 않을 태그 목록
- `denylist_tags`: 전처리 단계에서 제거할 태그
- `separators`: 링크를 보존할 때는 `"."` 대신 `". "` 처럼 공백을 포함한 구분자를 쓰세요. (URL의 점에서 잘리는 것을 방지)

In [ ]:
from langchain_text_splitters import HTMLSemanticPreservingSplitter

html_with_table = """
<!DOCTYPE html>
<html>
<body>
    <h1>상품 안내</h1>
    <p>이 섹션에는 여러 청크로 나뉘면 안 되는 중요한 표와 목록이 있습니다.</p>
    <table>
        <tr><th>품목</th><th>수량</th><th>가격</th></tr>
        <tr><td>사과</td><td>10</td><td>1,000원</td></tr>
        <tr><td>오렌지</td><td>5</td><td>500원</td></tr>
        <tr><td>바나나</td><td>50</td><td>1,500원</td></tr>
    </table>
    <h2>세부 사항</h2>
    <p>아래 목록은 하나의 청크에 함께 있어야 의미가 통합니다.</p>
    <ul>
        <li>항목 1: 매우 상세하고 중요한 첫 번째 항목 설명입니다.</li>
        <li>항목 2: 중요한 정보를 담고 있는 두 번째 항목 설명입니다.</li>
        <li>항목 3: 여러 청크로 나뉘지 않아야 하는 세 번째 항목 설명입니다.</li>
    </ul>
</body>
</html>
"""

semantic_splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=[("h1", "Header 1"), ("h2", "Header 2")],
    max_chunk_size=50,                      # 일부러 작게 설정하여 보존 효과를 확인
    separators=["\n\n", "\n", ". ", "! ", "? "],
    elements_to_preserve=["table", "ul"],   # 표와 목록은 통째로 유지
    denylist_tags=["script", "style", "head"],
)

for doc in semantic_splitter.split_text(html_with_table):
    print(doc.page_content)
    print(doc.metadata, end="\n=====================\n")